In [ ]:
import torch
import os

# --- FIX: FORCE INITIALIZATION FOR TORCH._DYNAMO ---
try:
    import torch._dynamo
    import torch._dynamo.decorators
except (ImportError, AttributeError):
    pass
# --------------------------------------------------

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import random
from collections import deque

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps) dengan amplifikasi realistis.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 
        
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                            
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            
                            # Scaling (Max ~12 Mbps untuk memicu kompetisi kualitas)
                            max_tp = max(throughput_mbps) if throughput_mbps else 1
                            scale_factor = 12.0 / max_tp if max_tp > 0 else 1.0
                            
                            scaled_mbps = [max(0.1, tp * scale_factor) for tp in throughput_mbps]
                            # Pre-smoothing trace dasar
                            smoothed = pd.Series(scaled_mbps).rolling(window=3, min_periods=1).mean().tolist()
                            
                            self.traces.append({
                                "name": file,
                                "data": smoothed
                            })
                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")
        
        if not self.traces:
            print("⚠️ traces_folder tidak ditemukan. Gunakan fallback.")
            self.traces.append({"name": "synth", "data": [8.0] * 500})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class HybridStreamingEnvNDN(gym.Env):
    """
    Updated Hybrid ABR Architecture: NDN-Aware RL + Dynamic Controller.
    Sesuai dengan integrasi shaka-ndn-plugin.
    """
    def __init__(self, trace_manager):
        super(HybridStreamingEnvNDN, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0]

        # Observation Space diperluas menjadi 7 dimensi:
        # [Buffer, Mean_TP, Last_Exec, Buffer_Safety, Volatility, CWND, RTT]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),
            high=np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]),
            dtype=np.float32
        )

        self.action_space = spaces.Discrete(3)
        self.state = None
        self.max_steps = 100
        self.current_step = 0
        self.tp_history = deque(maxlen=10)

        # NDN Constants
        self.LOW_BUFFER_THRESHOLD = 5.0
        self.PANIC_BUFFER_THRESHOLD = 3.0
        self.SAFE_MARGIN = 1.5
        self.UPGRADE_CONFIRM_STEPS = 2
        
        # NDN Thresholds
        self.NDN_CONGESTION_CWND = 5.0  # Batas kritis CWND
        self.NDN_CACHE_HIT_RTT = 20.0   # Batas RTT dianggap Cache Hit (ms)

    def _get_normalized_obs(self):
        # Destructure state (7 dimensi)
        buffer, mean_tp, last_exec, _, volatility, cwnd, rtt = self.state

        # Normalisasi fitur konvensional
        norm_buffer = np.clip(buffer / 30.0, 0.0, 1.0)
        norm_tp = np.clip(mean_tp / 10.0, 0.0, 1.0)
        norm_exec = float(last_exec) / 2.0
        safety = max(0, buffer - self.LOW_BUFFER_THRESHOLD)
        norm_safety = np.clip(safety / 15.0, 0.0, 1.0)
        norm_volatility = np.clip(volatility, 0.0, 1.0)

        # --- NORMALISASI FITUR NDN ---
        norm_cwnd = np.clip(cwnd / 100.0, 0.0, 1.0)  # Skala 0-100 paket
        norm_rtt = np.clip(rtt / 500.0, 0.0, 1.0)    # Skala 0-500 ms

        return np.array([
            norm_buffer, norm_tp, norm_exec, norm_safety, 
            norm_volatility, norm_cwnd, norm_rtt
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        trace_name = self.trace_manager.select_random_trace()
        initial_tp = self.trace_manager.get_next_bandwidth()
        
        self.tp_history.clear()
        for _ in range(self.tp_history.maxlen):
            self.tp_history.append(initial_tp)

        # Initial State: [buffer, tp, exec, safety, vol, cwnd, rtt]
        # Kita mulai dengan CWND 10 dan RTT 100ms sebagai default
        self.state = np.array([15.0, initial_tp, 1.0, 0.0, 0.0, 10.0, 100.0], dtype=np.float32)
        self.current_step = 0
        return self._get_normalized_obs(), {"trace": trace_name}

    def step(self, action):
        target_idx = int(action)
        buffer, _, current_idx, _, _, prev_cwnd, prev_rtt = self.state
        current_idx = int(current_idx)

        # 1. Update Throughput & Metrik Jaringan
        raw_tp = self.trace_manager.get_next_bandwidth()
        self.tp_history.append(raw_tp)
        mean_tp = np.mean(self.tp_history)
        volatility = np.std(self.tp_history) / (mean_tp + 1e-6)

        # SIMULASI METRIK NDN (Dalam produksi, ini didapat dari shaka-ndn-plugin)
        # cwnd biasanya proporsional dengan throughput, rtt kebalikannya
        new_cwnd = np.clip(raw_tp * 8.0 + np.random.normal(0, 2), 2, 100)
        new_rtt = np.clip(200.0 / (raw_tp + 0.5) + np.random.normal(0, 5), 5, 500)

        # --- LAYER 2: NDN-AWARE DYNAMIC CONTROLLER (INTERVENSI) ---
        executed_idx = current_idx
        dynamic_buffer_req = self.LOW_BUFFER_THRESHOLD + (volatility * 5.0)

        # A. NDN VETO LOGIC (Intervensi Utama)
        # Jika CWND sangat rendah (jalur macet), AI dilarang keras menaikkan bitrate
        is_ndn_congested = new_cwnd < self.NDN_CONGESTION_CWND
        
        if is_ndn_congested and target_idx > current_idx:
            target_idx = current_idx # Batalkan rencana upgrade AI
            # print(f"DEBUG: NDN Veto Upgrade! CWND: {new_cwnd:.2f}")

        # B. Panic Mode (Buffer Kritis)
        if buffer < self.PANIC_BUFFER_THRESHOLD:
            executed_idx = max(0, current_idx - 1)
        
        # C. Upgrade Logic dengan Hysteresis & NDN Health
        elif target_idx > current_idx:
            headroom = mean_tp - self.bitrates[current_idx]
            # Upgrade hanya jika margin aman, buffer cukup, DAN RTT tidak sedang melonjak
            if headroom > self.SAFE_MARGIN and buffer > dynamic_buffer_req and new_rtt < 300:
                executed_idx = current_idx + 1
            else:
                executed_idx = current_idx
        
        # D. Downgrade Logic
        else:
            executed_idx = target_idx

        # 2. Eksekusi Fisik & Hitung Stalling
        executed_idx = np.clip(executed_idx, 0, len(self.bitrates) - 1)
        chosen_bitrate = self.bitrates[executed_idx]
        
        # DASH Download Simulation
        download_time = (chosen_bitrate * 5.0 / (raw_tp + 0.1))
        stalling = max(0, download_time - buffer)
        new_buffer = max(0, buffer - download_time) + 5.0 # 5s per segment
        new_buffer = min(new_buffer, 30.0)

        # 3. Reward Function (Menghukum ketidakstabilan & stalling)
        reward = float(chosen_bitrate)
        if stalling > 0:
            reward -= (stalling * 50.0 + 20.0) # Penalti berat untuk buffering
        if executed_idx != current_idx:
            reward -= 2.0 # Penalti untuk fluktuasi kualitas

        # Update Internal State (7 Dimensi)
        self.state = np.array([
            new_buffer, mean_tp, float(executed_idx), 0.0, 
            volatility, new_cwnd, new_rtt
        ], dtype=np.float32)

        self.current_step += 1
        done = self.current_step >= self.max_steps

        return self._get_normalized_obs(), reward, done, False, {
            "buffer": new_buffer,
            "cwnd": new_cwnd,
            "rtt": new_rtt,
            "bitrate": chosen_bitrate,
            "raw_tp": mean_tp,        # Tambahkan ini agar line 272 tidak error
            "volatility": volatility  # Tambahkan ini agar line 273 tidak error
        }
def run_experiment():
    log_dir = "./rl_logs/"
    os.makedirs(log_dir, exist_ok=True)
    
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces: return
        
    env = Monitor(HybridStreamingEnvNDN(tm), log_dir)
    
    print("⏳ Memulai pelatihan 'Volatility-Aware Hybrid ABR' (300,000 langkah)...")
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.00025,  
                ent_coef=0.05, 
                n_steps=2048,
                batch_size=64)         
    
    model.learn(total_timesteps=300000)
    model.save("volatility_hybrid_ndn_model")
    print("✅ Pelatihan selesai.")

    # EVALUASI 8 FILE
    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            action_val = action.item() if isinstance(action, np.ndarray) else int(action)
            history.append({
                'TP': info_step['raw_tp'],
                'Volatility': info_step['volatility'],
                'Buffer': info_step['buffer'], 
                'Exec_Bitrate': info_step['bitrate']
            })
            
        df = pd.DataFrame(history)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Grafik Bitrate + Throughput
        ax1.plot(df.index, df['TP'], label='Raw TP', color='blue', alpha=0.15)
        ax1.step(df.index, df['Exec_Bitrate'], label='Executed Bitrate', color='red', linewidth=2, where='post')
        ax1.set_title(f"Volatility-Aware: {trace_name}")
        ax1.set_ylabel("Mbps")
        ax1.legend()
        
        # Grafik Buffer + Volatility
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.15, label='Buffer Level')
        ax2_vol = ax2.twinx()
        ax2_vol.plot(df.index, df['Volatility'], label='Volatility (CV)', color='purple', linestyle=':', alpha=0.6)
        ax2_vol.set_ylabel("Volatility Scale")
        ax2.axhline(y=5, color='orange', linestyle='--', label='Base Safe')
        ax2.set_xlabel("Segmen")
        ax2.legend(loc='upper right')
        
        plt.tight_layout()
        plt.savefig(os.path.join(log_dir, f"vol_hybrid_{trace_name}.png"))
        plt.close()

if __name__ == "__main__":
    run_experiment()

⏳ Memulai pelatihan 'Volatility-Aware Hybrid ABR' (300,000 langkah)...
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -137     |
| time/              |          |
|    fps             | 3090     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -40.9       |
| time/                   |             |
|    fps                  | 2435        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012851896 |
|    clip_fraction        | 0.181       |
|    clip_range           | 0.2  

KeyboardInterrupt: 